<a href="https://colab.research.google.com/github/Saty290501/text-to-video-colab/blob/main/text_to_video_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q "diffusers==0.35.1" "transformers>=4.49,<5" accelerate imageio-ffmpeg

In [3]:
import torch
import diffusers
import transformers
import huggingface_hub

from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Pipeline imports passed")

Diffusers: 0.35.1
Transformers: 4.57.6
Hugging Face Hub: 0.36.2
GPU: Tesla T4
Pipeline imports passed


In [ ]:
model_id = "cerspense/zeroscope_v2_576w"

pipe = DiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

print("Zeroscope is ready")

In [8]:
from time import perf_counter
from IPython.display import Video, display

prompt = ("A small yellow paper boat floats slowly from left to right across a shallow pond. Gentle ripples trail behind it. Fixed camera at water level.")
seed = 42
steps = 25

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = perf_counter()

frames = pipe(
    prompt=prompt,
    width=576,
    height=320,
    num_frames=24,
    num_inference_steps=steps,
    generator=torch.Generator(device="cpu").manual_seed(seed),
).frames[0]

torch.cuda.synchronize()
elapsed = perf_counter() - start
peak_gib = torch.cuda.max_memory_allocated() / 2**30

filename = "boat.mp4"
export_to_video(frames, filename, fps=8)
display(Video(filename, embed=True))

print(f"Generated {len(frames)} frames in {elapsed:.1f} seconds")
print(f"Peak GPU memory allocated by PyTorch: {peak_gib:.2f} GiB")
display(Video(filename, embed=True))

  0%|          | 0/25 [00:00<?, ?it/s]

Generated 24 frames in 99.4 seconds
Peak GPU memory allocated by PyTorch: 5.39 GiB
